# Direction Y2: Direction O modules dưới kiểm định thống kê + lineage-aware

**Mục tiêu:** Trả lời: *các module nâng cao của Direction O (ensemble selector, functional-like, sample-graph, gene-graph embedding) có thật sự đánh bại paper-ready 50 một cách CÓ Ý NGHĨA THỐNG KÊ không?*

Port **toàn bộ module của Direction O** vào cùng harness (repeated stratified CV + corrected resampled *t*-test + Holm), chọn adaptive-best trong nhóm module nâng cao, so với paper-ready 50, và thêm lineage-aware + negative control.


## 0. Imports + cấu hình

In [ ]:
import os, re, json, math, time, shutil, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display
from scipy import stats as sstats, sparse
from sklearn.model_selection import StratifiedKFold, GroupKFold, train_test_split
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import balanced_accuracy_score, f1_score, average_precision_score
import matplotlib.pyplot as plt
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
print("HAS_XGB:", HAS_XGB)

In [ ]:
REPO_URL = "https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git"
BASE_DIR = Path("/content/salmonella_direction_Y2_O_significance")
REPO_DIR = BASE_DIR / "Antimicrobial-resistance-prediction-in-Salmonella"
EXTRACT_DIR = BASE_DIR / "extracted"
OUTPUT_DIR = BASE_DIR / "outputs"
for d in [BASE_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRUGS = ["AMP", "AUG", "AXO", "CHL", "FOX"]
N_REPEATS, N_SPLITS, BASE_SEED = 5, 5, 42
RANDOM_SEEDS = list(range(BASE_SEED, BASE_SEED + N_REPEATS))
K_SMALL, K_MAIN, K_LARGE, MAXF = 50, 200, 500, 5000
MODELS = ["LR_balanced", "RF_balanced"] + (["XGB_weighted"] if HAS_XGB else [])
N_LINEAGE_CLUSTERS, ALPHA = 30, 0.05
print("MODELS:", MODELS, "| CV:", N_REPEATS, "x", N_SPLITS)

## 1. Clone repo + giải nén accessory matrix

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
!apt-get update -qq
!apt-get install -y unrar > /dev/null
acc_dir = EXTRACT_DIR / "accessory_gene"; acc_dir.mkdir(parents=True, exist_ok=True)
rar = REPO_DIR / "results" / "Roary" / "accessory gene existence matrix.rar"
if not any(acc_dir.glob("*")):
    if rar.exists():
        !unrar x -o+ "{rar}" "{acc_dir}/" > /dev/null
    else:
        lr = BASE_DIR / "acc.rar"
        url = "https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella/raw/main/results/Roary/accessory%20gene%20existence%20matrix.rar"
        !wget -q -O "{lr}" "{url}"
        !unrar x -o+ "{lr}" "{acc_dir}/" > /dev/null
!find "{acc_dir}" -maxdepth 2 -type f | head

## 2. Nạp dữ liệu

In [ ]:
def cn(df):
    o = df.copy(); drop = []
    for c in list(o.columns):
        if o[c].dtype == "object":
            v = pd.to_numeric(o[c], errors="coerce")
            if v.notna().mean() > 0.95: o[c] = v.fillna(0)
            else: drop.append(c)
    return o.drop(columns=drop).fillna(0)

def pl(y):
    if isinstance(y, pd.DataFrame): y = y[y.columns[-1]]
    return pd.to_numeric(y.replace({"S":0,"I":0,"R":1,"s":0,"i":0,"r":1,
        "Susceptible":0,"Intermediate":0,"Resistant":1,
        "susceptible":0,"intermediate":0,"resistant":1}), errors="coerce")

def find_largest(root):
    cs = [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in [".csv",".tsv",".txt",".xlsx"]]
    return sorted(cs, key=lambda p: p.stat().st_size, reverse=True)[0]

ACC = cn(pd.read_csv(find_largest(acc_dir)))
ACC.columns = [f"g{i}" for i in range(ACC.shape[1])]
print("accessory:", ACC.shape)

def load(drug):
    dd = REPO_DIR / "data" / "csv" / drug
    X = cn(pd.read_csv(dd / "gene.csv"))
    lf = dd / f"{drug}_label.csv"
    if not lf.exists(): lf = list(dd.glob("*label*.csv"))[0]
    yf = pl(pd.read_csv(lf)); m = yf.notna().values; pos = np.where(m)[0]
    return (X.loc[m].reset_index(drop=True), yf.loc[m].reset_index(drop=True).astype(int),
            ACC.iloc[pos].reset_index(drop=True))

ready = {d: load(d) for d in DRUGS}
display(pd.DataFrame([{"drug":d,"n":len(ready[d][1]),"n_R":int(ready[d][1].sum()),
    "resistant_rate":round(float(ready[d][1].mean()),4)} for d in DRUGS]))

## 3. Toàn bộ feature module của Direction O (fit trên train, áp cho val/test)

Bê nguyên logic Direction O: chi2/ensemble selector, functional-like keyword groups, sample-graph (Jaccard neighbor stats), gene-graph embedding (TruncatedSVD trên đồ thị đồng xuất hiện gene). Mọi selector/graph/embedding **chỉ fit trên tập train của fold**.


In [ ]:
def top_var_cols(Xdf, maxf=MAXF):
    if Xdf.shape[1] <= maxf: return list(Xdf.columns)
    return list(Xdf.var(0).sort_values(ascending=False).head(maxf).index)
def norm(s):
    s = s.astype(float).replace([np.inf,-np.inf],np.nan).fillna(0)
    return pd.Series(np.zeros(len(s)),index=s.index) if s.max()==s.min() else (s-s.min())/(s.max()-s.min())
def chi2_rank(Xc,y):
    sc,_=chi2(Xc.clip(lower=0),y); return pd.Series(sc,index=Xc.columns).replace([np.inf,-np.inf],np.nan).fillna(0)
def ensemble_rank(Xc,y,seed):
    scores=[norm(chi2_rank(Xc,y))]
    try:
        m=LogisticRegression(penalty="l1",solver="liblinear",class_weight="balanced",max_iter=1000,random_state=seed).fit(Xc,y)
        scores.append(norm(pd.Series(np.abs(m.coef_[0]),index=Xc.columns)))
    except Exception: pass
    try:
        m=RandomForestClassifier(n_estimators=120,min_samples_leaf=2,class_weight="balanced_subsample",random_state=seed,n_jobs=-1).fit(Xc,y)
        scores.append(norm(pd.Series(m.feature_importances_,index=Xc.columns)))
    except Exception: pass
    return pd.concat(scores,axis=1).mean(axis=1)

FUNC_GROUPS={"beta_lactamase":["cmy","tem","oxa","ctx","shv","bla","beta","lactamase"],
 "efflux_transporter":["flor","suge","tolc","mdfa","acr","efflux","transporter","pump"],
 "sulfonamide":["sul1","sul2","sul3","sul"],"metal_stress":["mera","mert","mer","ars","metal","copper","silver"],
 "mobile_element":["transposase","integrase","plasmid","insertion","tnp"],"phage_related":["phage","tail","capsid","bacteriophage"]}
def functional_like(Xdf):
    Xdf=pd.DataFrame(Xdf); cl={c:str(c).lower() for c in Xdf.columns}; out={}
    for g,kw in FUNC_GROUPS.items():
        mt=[c for c,x in cl.items() if any(k in x for k in kw)]
        cnt=Xdf[mt].sum(1).values if mt else np.zeros(Xdf.shape[0])
        out[f"func_{g}_count"]=cnt; out[f"func_{g}_presence"]=(cnt>0).astype(int)
    return pd.DataFrame(out,index=Xdf.index)
def jac(A,B):
    A=np.asarray(A,np.float32);B=np.asarray(B,np.float32);inter=A@B.T
    return inter/np.maximum(A.sum(1,keepdims=True)+B.sum(1,keepdims=True).T-inter,1e-8)
def sample_graph(Xtr_sel,ytr,Xq_sel,is_train,top_k=15):
    sim=jac(Xq_sel.values,Xtr_sel.values); ytr=np.asarray(ytr,float)
    if is_train and sim.shape[0]==sim.shape[1]: np.fill_diagonal(sim,-1.0)
    feats=[]
    for i in range(sim.shape[0]):
        row=sim[i].copy(); idx=np.argsort(row)[::-1][:top_k]; sims=row[idx];labels=ytr[idx]
        v=sims>=0; sims=sims[v];labels=labels[v]
        if len(sims)==0: feats.append([0,0,0,0,0,0,0]);continue
        w=np.maximum(sims,0); wr=float((w*labels).sum()/w.sum()) if w.sum()>0 else float(labels.mean())
        rs=row[ytr==1]; ss=row[ytr==0]
        feats.append([float(labels.mean()),wr,float(np.max(sims)),float(np.mean(sims)),
                      float(np.max(rs)) if len(rs) else 0.0,float(np.max(ss)) if len(ss) else 0.0,float((labels==1).sum())])
    cols=["nbr_R_rate","w_nbr_R_rate","max_sim","mean_topk_sim","max_sim_R","max_sim_S","topk_R_cnt"]
    return pd.DataFrame(feats,index=Xq_sel.index,columns=cols)
def gene_emb_fit(Xtr_sel,n=16):
    X=Xtr_sel.astype(float); corr=np.nan_to_num(np.corrcoef(X.values,rowvar=False)); np.fill_diagonal(corr,0);corr=np.abs(corr)
    A=np.zeros_like(corr,np.float32)
    for i in range(corr.shape[0]):
        idx=np.argsort(corr[i])[::-1][:8]; A[i,idx]=corr[i,idx]
    A=np.maximum(A,A.T); n=min(n,max(2,A.shape[0]-1))
    return TruncatedSVD(n_components=n,random_state=42).fit_transform(A)
def gene_emb_transform(Xsel,emb):
    Xv=Xsel.values.astype(float); cnt=np.maximum(Xv.sum(1,keepdims=True),1.0)
    out=np.hstack([(Xv@emb)/cnt, Xv.sum(1,keepdims=True)])
    return pd.DataFrame(out,index=Xsel.index,columns=[f"emb_{i}" for i in range(emb.shape[1])]+["sel_gene_cnt"])
def cat(*dfs): return pd.concat([pd.DataFrame(d).reset_index(drop=True) for d in dfs],axis=1)

def build_modules(Xa_tr,Xr_tr,ytr,Xa_va,Xr_va,Xa_te,Xr_te,seed):
    mods={}
    mods["paper_ready50"]=(Xr_tr,Xr_va,Xr_te)
    cand=top_var_cols(Xa_tr)
    ch=chi2_rank(Xa_tr[cand],ytr).sort_values(ascending=False)
    c200=list(ch.head(K_MAIN).index); c500=list(ch.head(K_LARGE).index)
    A2=(Xa_tr[c200],Xa_va[c200],Xa_te[c200]); A5=(Xa_tr[c500],Xa_va[c500],Xa_te[c500])
    mods["accessory_chi2_200"]=A2; mods["accessory_chi2_500"]=A5
    mods["paper_ready50_plus_chi2_200"]=(cat(Xr_tr,A2[0]),cat(Xr_va,A2[1]),cat(Xr_te,A2[2]))
    mods["paper_ready50_plus_chi2_500"]=(cat(Xr_tr,A5[0]),cat(Xr_va,A5[1]),cat(Xr_te,A5[2]))
    er=ensemble_rank(Xa_tr[cand],ytr,seed).sort_values(ascending=False)
    e50=list(er.head(K_SMALL).index); e500=list(er.head(K_LARGE).index)
    mods["ensemble_top_50"]=(Xa_tr[e50],Xa_va[e50],Xa_te[e50])
    mods["ensemble_top_500"]=(Xa_tr[e500],Xa_va[e500],Xa_te[e500])
    Ftr,Fva,Fte=functional_like(A2[0]),functional_like(A2[1]),functional_like(A2[2])
    mods["accessory200_plus_functional_like"]=(cat(A2[0],Ftr),cat(A2[1],Fva),cat(A2[2],Fte))
    Gtr=sample_graph(A2[0],ytr,A2[0],True); Gva=sample_graph(A2[0],ytr,A2[1],False); Gte=sample_graph(A2[0],ytr,A2[2],False)
    mods["accessory200_plus_sample_graph"]=(cat(A2[0],Gtr),cat(A2[1],Gva),cat(A2[2],Gte))
    mods["sample_graph_only"]=(Gtr,Gva,Gte)
    emb=gene_emb_fit(A2[0]); Etr,Eva,Ete=gene_emb_transform(A2[0],emb),gene_emb_transform(A2[1],emb),gene_emb_transform(A2[2],emb)
    mods["accessory200_plus_gene_graph_embedding"]=(cat(A2[0],Etr),cat(A2[1],Eva),cat(A2[2],Ete))
    mods["gene_graph_embedding_only"]=(Etr,Eva,Ete)
    return mods

## 4. Model + đánh giá 1 fold

In [ ]:
def mk(name, ytr, seed):
    if name=="LR_balanced": return LogisticRegression(max_iter=5000,class_weight="balanced",random_state=seed)
    if name=="RF_balanced": return RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=seed)
    pos=max(int((ytr==1).sum()),1); neg=max(int((ytr==0).sum()),1)
    return xgb.XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,subsample=0.9,
        colsample_bytree=0.9,eval_metric="logloss",scale_pos_weight=neg/pos,random_state=seed,n_jobs=-1,use_label_encoder=False)
def thr_pick(yv,pv):
    bt,bs=0.5,-1
    for t in np.linspace(0.05,0.95,91):
        s=f1_score(yv,(pv>=t).astype(int),zero_division=0)
        if s>bs: bs,bt=s,float(t)
    return bt
def fit_eval(Xtr,ytr,Xva,yva,Xte,yte,name,seed):
    Xtr=np.nan_to_num(pd.DataFrame(Xtr).values.astype(float)); Xva=np.nan_to_num(pd.DataFrame(Xva).values.astype(float)); Xte=np.nan_to_num(pd.DataFrame(Xte).values.astype(float))
    if len(np.unique(ytr))<2: return None
    m=mk(name,ytr,seed); m.fit(Xtr,ytr); thr=thr_pick(yva,m.predict_proba(Xva)[:,1])
    m=mk(name,ytr,seed); m.fit(Xtr,ytr); pr=m.predict_proba(Xte)[:,1]
    return {"f1":f1_score(yte,(pr>=thr).astype(int),zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(yte,(pr>=thr).astype(int)),
            "auprc":average_precision_score(yte,pr) if len(np.unique(yte))>1 else np.nan}

## 5. Repeated stratified CV cho toàn bộ module (mất thời gian nhất)

In [ ]:
t0=time.time(); rec=[]
for drug in DRUGS:
    Xr,y,Xa=ready[drug]; y=np.asarray(y)
    for rep,seed in enumerate(RANDOM_SEEDS):
        skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed)
        for fold,(tr,te) in enumerate(skf.split(np.zeros(len(y)),y)):
            itr,iva=train_test_split(np.arange(len(tr)),test_size=0.25,stratify=y[tr],random_state=seed+fold+1000)
            fi,vi=tr[itr],tr[iva]
            mods=build_modules(Xa.iloc[fi],Xr.iloc[fi],y[fi],Xa.iloc[vi],Xr.iloc[vi],Xa.iloc[te],Xr.iloc[te],seed)
            for name,(Mtr,Mva,Mte) in mods.items():
                for model in MODELS:
                    sc=fit_eval(Mtr,y[fi],Mva,y[vi],Mte,y[te],model,seed)
                    if sc: rec.append({"drug":drug,"module":name,"model":model,"repeat":rep,"fold":fold,**sc})
    print("CV done",drug,round(time.time()-t0,1),"s",flush=True)
fold_df=pd.DataFrame(rec); fold_df.to_csv(OUTPUT_DIR/"Y2_per_fold.csv",index=False)
agg=fold_df.groupby(["drug","module","model"])[["f1","balanced_accuracy","auprc"]].mean().reset_index()
agg.to_csv(OUTPUT_DIR/"Y2_mean_scores.csv",index=False)
display(agg.sort_values(["drug","f1"],ascending=[True,False]).groupby("drug").head(3))

## 6. Kiểm định thống kê: adaptive-best (module nâng cao) vs paper_ready50

In [ ]:
ADV=[m for m in fold_df.module.unique() if m!="paper_ready50"]
def best_cfg(sub,mods):
    g=sub[sub.module.isin(mods)].groupby(["module","model"]).f1.mean().reset_index()
    r=g.sort_values("f1",ascending=False).iloc[0]; return r["module"],r["model"]
def corr_ttest(d,ntr,nte):
    d=np.asarray(d,float);n=len(d);md_=d.mean();vd=d.var(ddof=1)
    if vd==0: return (np.inf if md_!=0 else 0.0),(0.0 if md_!=0 else 1.0)
    t=md_/math.sqrt(vd*((1.0/n)+(nte/ntr))); return float(t),float(2*sstats.t.sf(abs(t),df=n-1))
def bci(d,nb=5000,seed=0):
    rng=np.random.RandomState(seed);d=np.asarray(d,float)
    ms=[d[rng.randint(0,len(d),len(d))].mean() for _ in range(nb)]; return float(np.percentile(ms,2.5)),float(np.percentile(ms,97.5))

nt=len(ready[DRUGS[0]][1]); nte=nt/N_SPLITS; ntr=nt-nte; cr=[]
for drug in DRUGS:
    sub=fold_df[fold_df.drug==drug]
    bmod,bmodel=best_cfg(sub,["paper_ready50"]); amod,amodel=best_cfg(sub,ADV)
    base=sub[(sub.module==bmod)&(sub.model==bmodel)].sort_values(["repeat","fold"])
    adap=sub[(sub.module==amod)&(sub.model==amodel)].sort_values(["repeat","fold"])
    mg=base.merge(adap,on=["repeat","fold"],suffixes=("_b","_a")); d=(mg.f1_a-mg.f1_b).values
    t,p=corr_ttest(d,ntr,nte)
    try: _,wp=sstats.wilcoxon(mg.f1_a,mg.f1_b)
    except Exception: wp=np.nan
    lo,hi=bci(d,seed=BASE_SEED)
    cr.append({"drug":drug,"baseline":f"{bmod}/{bmodel}","adaptive_best":f"{amod}/{amodel}",
        "f1_base":round(base.f1.mean(),4),"f1_adapt":round(adap.f1.mean(),4),"delta_f1":round(d.mean(),4),
        "ci95_lo":round(lo,4),"ci95_hi":round(hi,4),"t_corr":round(t,3),"p_corr":p,"wilcoxon_p":wp})
comp=pd.DataFrame(cr)
order=comp.p_corr.fillna(1).values.argsort(); m=len(comp); holm=np.empty(m); run=0.0
for rk,idx in enumerate(order):
    adj=min((m-rk)*comp.p_corr.fillna(1).values[idx],1.0); run=max(run,adj); holm[idx]=run
comp["p_holm"]=holm; comp["sig_holm"]=comp.p_holm<ALPHA
comp.to_csv(OUTPUT_DIR/"Y2_statistical_comparison.csv",index=False)
display(comp)
print("Số thuốc adaptive>baseline có ý nghĩa (Holm<%.2f): %d/%d"%(ALPHA,int(comp.sig_holm.sum()),len(comp)))

## 7. Lineage-aware (leave-cluster-out) cho adaptive-best

In [ ]:
def jac_dist(Xb):
    Xs=sparse.csr_matrix((Xb>0).astype(np.uint8)); inter=(Xs@Xs.T).toarray().astype(np.float32)
    rs=np.asarray(Xs.sum(1)).ravel().astype(np.float32); union=rs[:,None]+rs[None,:]-inter
    sim=np.divide(inter,union,out=np.zeros_like(inter),where=union>0); dist=1.0-sim; np.fill_diagonal(dist,0)
    iu=np.triu_indices(len(dist),1); return dist, float(sim[iu].max())
lin=[]
for drug in DRUGS:
    Xr,y,Xa=ready[drug]; y=np.asarray(y)
    amod,amodel=best_cfg(fold_df[fold_df.drug==drug],ADV)
    dist,maxj=jac_dist(Xa.values)
    K=min(N_LINEAGE_CLUSTERS,len(y)-1)
    lab=AgglomerativeClustering(n_clusters=K,metric="precomputed",linkage="average").fit_predict(dist)
    rnd=fold_df[(fold_df.drug==drug)&(fold_df.module==amod)&(fold_df.model==amodel)].f1
    gkf=GroupKFold(min(N_SPLITS,len(np.unique(lab)))); lf=[]
    for tr,te in gkf.split(np.zeros(len(y)),y,groups=lab):
        if len(np.unique(y[te]))<2 or len(np.unique(y[tr]))<2: continue
        itr,iva=train_test_split(np.arange(len(tr)),test_size=0.25,stratify=y[tr],random_state=BASE_SEED)
        fi,vi=tr[itr],tr[iva]
        mods=build_modules(Xa.iloc[fi],Xr.iloc[fi],y[fi],Xa.iloc[vi],Xr.iloc[vi],Xa.iloc[te],Xr.iloc[te],BASE_SEED)
        Mtr,Mva,Mte=mods[amod]; sc=fit_eval(Mtr,y[fi],Mva,y[vi],Mte,y[te],amodel,BASE_SEED)
        if sc: lf.append(sc["f1"])
    lin.append({"drug":drug,"config":f"{amod}/{amodel}","max_pairwise_jaccard":round(maxj,4),"n_clusters":int(K),
        "random_f1":round(float(rnd.mean()),4),"lineage_f1":round(float(np.mean(lf)),4) if lf else np.nan,
        "f1_drop":round(float(rnd.mean()-np.mean(lf)),4) if lf else np.nan})
    print("lineage done",drug,flush=True)
lineage=pd.DataFrame(lin); lineage.to_csv(OUTPUT_DIR/"Y2_lineage_aware.csv",index=False); display(lineage)

## 8. Negative control (shuffled labels)

In [ ]:
neg=[]; rng=np.random.RandomState(BASE_SEED)
for drug in DRUGS:
    Xr,y,Xa=ready[drug]; y=np.asarray(y); amod,amodel=best_cfg(fold_df[fold_df.drug==drug],ADV)
    ys=y.copy(); rng.shuffle(ys); skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=BASE_SEED); ba=[]
    for tr,te in skf.split(np.zeros(len(ys)),ys):
        itr,iva=train_test_split(np.arange(len(tr)),test_size=0.25,stratify=ys[tr],random_state=BASE_SEED)
        fi,vi=tr[itr],tr[iva]
        mods=build_modules(Xa.iloc[fi],Xr.iloc[fi],ys[fi],Xa.iloc[vi],Xr.iloc[vi],Xa.iloc[te],Xr.iloc[te],BASE_SEED)
        Mtr,Mva,Mte=mods[amod]; sc=fit_eval(Mtr,ys[fi],Mva,ys[vi],Mte,ys[te],amodel,BASE_SEED)
        if sc: ba.append(sc["balanced_accuracy"])
    neg.append({"drug":drug,"config":f"{amod}/{amodel}","neg_control_bal_acc":round(float(np.mean(ba)),4)})
negdf=pd.DataFrame(neg); negdf.to_csv(OUTPUT_DIR/"Y2_negative_control.csv",index=False); display(negdf)

## 9. Hình + gom output

In [ ]:
fig,ax=plt.subplots(figsize=(7,4)); xp=np.arange(len(comp))
ax.bar(xp,comp.delta_f1,yerr=[comp.delta_f1-comp.ci95_lo,comp.ci95_hi-comp.delta_f1],capsize=5,
       color=["#2a9d8f" if s else "#adb5bd" for s in comp.sig_holm])
ax.axhline(0,color="k",lw=0.8); ax.set_xticks(xp); ax.set_xticklabels(comp.drug)
ax.set_ylabel("Δ F1 (O adaptive-best − paper_ready50)")
ax.set_title("Direction O modules vs paper-ready 50 (bootstrap 95% CI)\nxanh = có ý nghĩa Holm")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"Y2_fig1_delta_f1.png",dpi=150); plt.show()

fig,ax=plt.subplots(figsize=(7,4)); w=0.38; xp=np.arange(len(lineage))
ax.bar(xp-w/2,lineage.random_f1,w,label="random CV",color="#264653")
ax.bar(xp+w/2,lineage.lineage_f1,w,label="lineage-aware",color="#e76f51")
ax.set_xticks(xp); ax.set_xticklabels(lineage.drug); ax.set_ylim(0,1); ax.set_ylabel("F1")
ax.set_title("Random vs lineage-aware (O adaptive-best)"); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"Y2_fig2_lineage.png",dpi=150); plt.show()

md_out=OUTPUT_DIR/"Y2_SUMMARY.md"
md_out.write_text("# Direction Y2 — O modules under statistical rigor\n\n## Statistical comparison\n"
    +comp.to_markdown(index=False)+"\n\n## Lineage-aware\n"+lineage.to_markdown(index=False)
    +"\n\n## Negative control\n"+negdf.to_markdown(index=False),encoding="utf-8")
shutil.make_archive(str(BASE_DIR/"direction_Y2_outputs"),"zip",OUTPUT_DIR)
print("Saved:",md_out)
try:
    from google.colab import files; files.download(str(BASE_DIR/"direction_Y2_outputs.zip"))
except Exception as e: print("skip download:",e)

## 10. Cách viết vào khóa luận

- **Nếu adaptive-best có ý nghĩa (Holm < 0.05) ở một số thuốc:** đó là bằng chứng định lượng cho luận điểm Direction O — viết "adaptive feature fusion cải thiện F1 có ý nghĩa thống kê ở N/5 thuốc sau hiệu chỉnh đa so sánh; module thắng khác nhau theo thuốc".
- **Nếu không có ý nghĩa:** viết trung thực "các module nâng cao ngang bằng paper-ready 50 về mặt thống kê; đóng góp chính là cho thấy accessory-based selection đạt hiệu năng tương đương với ít marker do chuyên gia chọn" — đây vẫn là kết quả khoa học hợp lệ và tránh overclaim.
- Bảng `Y2_statistical_comparison.csv` + Fig1 vào chương *Results*; `Y2_lineage_aware.csv` + Fig2 và `Y2_negative_control.csv` vào *Robustness*.
- So sánh trực tiếp với Direction Y (module đơn giản) để cho thấy tác động của module nâng cao.
